# TP3b — Sigma levels: a short look at a regional model, CROCO

**Prerequisites**: Course 2 (numerical models), TP3a (NEMO).

In TP3a you used a **global** model, NEMO at 1°, with **z-levels**: fixed depths, the same everywhere, cut by the sea floor as a staircase. Regional and coastal models often use another vertical grid: **sigma levels**, which **follow the sea floor**. This short notebook has **one goal**: understand what sigma levels are, and what they change when you **use** the outputs (a surface map, a section, a map at a given depth). The model is **CROCO**, a regional model of the North-East Tropical Pacific at ~5.5 km.

## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete**. Everything else (imports, helper functions, plot formatting) is already written.
- After each exercise, a **✅ Check** box tells you what a correct result should look like — use it to self-correct before moving on.
- Stuck on a `# TODO`? Below each exercise, a **💡 Hint** box (and, as a last resort, a **🔑 Solution** box) can be opened with a click. Try for a few minutes before opening them!
- **🚀 Advanced track** blocks are optional: if you finish an exercise before the others, try them while the instructor helps the rest of the group.

> **⚠️ Before you start: work on your own copy.** In the JupyterLab file browser, right-click this notebook → *Duplicate*, then rename the copy to `03b_croco_EN_mine.ipynb` and open it. Work only in your `_mine` copy, so the original stays clean and you can always go back to it.

## Reminder from TP3a

| **Model** | **TP3a — NEMO** | **TP3b — CROCO** |
|---|---|---|
| Domain | **global** | **regional**: North-East Tropical Pacific |
| Horizontal grid | 1° (≈ 110 km) | ≈ 5.5 km |
| Vertical levels | **z-levels**: fixed depths, the same everywhere | **sigma levels**: they follow the sea floor |

## Contents
- [0. Setup](#0-setup)
- [1. A first look at the file: where is the depth?](#1-anatomy)
- [A. The surface: the last level](#part-a)
- **[Part B — Sigma levels vs z-levels](#part-b)**
  - [B.1 Where are the model levels? CROCO vs NEMO](#b-1)
  - [B.2 A vertical section across the Costa Rica Dome](#b-2)
  - [B.3 A map at 100 m: interpolate, not select](#b-3)
  - [🚀 Advanced track A — the depth of the thermocline](#advanced-a)
  - [🚀 Advanced track B — CROCO vs the GLORYS12 reanalysis: the Dome section](#advanced-b)
- [C. 🐞 Debugging exercise](#c-debugging)
- [D. Recap, in pairs: global or regional?](#d-recap)

---
<a id="0-setup"></a>
## 0. Setup — imports, paths and helper functions

Run this cell as is. It imports a few **helper functions** from **`croco_helpers.py`**, next to this notebook (written with xarray only; open the file if you are curious):

| Function | What it does |
|---|---|
| `open_croco(avg_file, grd_file)` | opens a CROCO output file and fixes it (section 1) |
| `depth_levels(ds, zeta)` | depth (m, negative) of every model level (B.1) |
| `section_lat(da, lat)` | a vertical section along a latitude (B.2) |
| `at_depth(da, z, depth)` | interpolates each water column at a fixed depth, in m, negative (B.3) |
| `nearest_point(da, lon, lat)` | the value(s) at the grid point closest to (lon, lat) |
| `base_map(ax, title, extent=None)` | adds land, coastlines and a grid to a map (as in TP3a) |

⚠️ Keep your `_mine` copy **in this folder**: the notebook needs `croco_helpers.py` next to it.

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean

# Data already on Kabre (read-only)
OBS_DIR = "/data/gcambon/COMMONDATA/DATASETS_PSF/"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"

AVG_FILE = f"{OBS_DIR}/MODEL_DS/croco_avg_Y2023M12_light.nc"    # model outputs (daily averages)
GRD_FILE = f"{OBS_DIR}/MODEL_DS/croco_grd.nc"                   # model grid
NEMO_2D = f"{OBS_DIR}/MODEL_DS/NEMO/2D_Nemo4.2-PISCES_A_1_gr_20060101_20251231.nc"         # global model (TP3a)
NEMO_3D = f"{OBS_DIR}/MODEL_DS/NEMO/3D_Nemo4.2-PISCES_A_1_gr_20060101_20251231_extr2006.nc"
GLORYS_FILE = f"{OBS_DIR}/COSTARICA_DS/3Docean_costarica_daily_2023_12.nc"   # reanalysis (TP2a), advanced track B

from croco_helpers import open_croco, depth_levels, nearest_point, section_lat, at_depth, base_map

!ls -lh {OBS_DIR}/MODEL_DS

**✅ Check**: the `ls` shows `croco_avg_Y2023M12_light.nc` (~720 MB) and `croco_grd.nc` (~40 MB).

---
<a id="1-anatomy"></a>
## 1. A first look at the file: where is the depth?

Open the model file **the usual way**, as in TP3a, and look at it.

In [ ]:
ds_raw = xr.open_dataset(AVG_FILE, decode_times=False)
ds_raw

**🎯 Your turn**: from the output above,
1. What are the dimensions of `temp`? Is there a **depth** dimension, like `deptht` in NEMO?
2. What is the name of the vertical dimension? What are its first and last values?
3. What are the dimensions of `lon_rho`?

In [ ]:
print("temp dims:", ds_raw["temp"].dims)
print("s_rho: ", ds_raw.sizes["s_rho"], "levels, first / last value:", ..., ...)   # TODO
print("lon_rho dims:", ...)                                                         # TODO

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `float(ds_raw["s_rho"][0])` and `float(ds_raw["s_rho"][-1])`, as for `deptht` in TP3a.
- `ds_raw["lon_rho"].dims`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
print("s_rho: ", ds_raw.sizes["s_rho"], "levels, first / last value:", float(ds_raw["s_rho"][0]), float(ds_raw["s_rho"][-1]))
print("lon_rho dims:", ds_raw["lon_rho"].dims)
```

</details>

**✅ Check** — and what it means:

1. `temp` is `(time, s_rho, eta_rho, xi_rho)`: **no depth dimension**. In NEMO, level 10 was at the same depth everywhere; in CROCO it is not.
2. The vertical dimension is `s_rho`: **50 levels**, from **−0.99** to **−0.01**. It is **not a depth in metres** but a *fraction* of the water column: −1 is the sea floor, 0 is the surface. So level 10 is at ~30 m on a 100 m shelf and at ~1500 m offshore: these are **sigma levels**, they **follow the sea floor**. ⚠️ **Level 0 is at the bottom**, the last level (`s_rho=-1`) is at the surface: the reverse of NEMO.
3. `lon_rho` is **2D**, `(eta_rho, xi_rho)`: each grid point has its own longitude and latitude (a *curvilinear* grid). That is why maps use `x="lon_rho", y="lat_rho"` below.

💬 The raw file has two more traps (time in seconds since an origin not written in the file, and undeclared fill values on land tiles). We skip them: `open_croco()` fixes both, and takes the clean grid from `croco_grd.nc`. From now on we use it.

In [ ]:
ds = open_croco(AVG_FILE, GRD_FILE)
print(ds["time"].values)
print("lon_rho:", float(ds.lon_rho.min()), "to", float(ds.lon_rho.max()))
print("lat_rho:", float(ds.lat_rho.min()), "to", float(ds.lat_rho.max()))
print("Grid spacing: %.1f to %.1f km" % (float(1 / ds.pm.max()) / 1e3, float(1 / ds.pm.min()) / 1e3))

**✅ Check**: 3 daily averages, **1, 2 and 3 December 2023** (at noon, the middle of each day). The domain covers **104.5°W–77°W, 2.3°N–19.4°N** (from southern Mexico to Colombia), with a grid spacing of **~5.2 to 5.6 km**.

💬 Compare with the products of TP1b/TP2a: OSTIA (0.05° ≈ 5.5 km), GLORYS12 (1/12° ≈ 9 km), WOA (1° ≈ 110 km).

💬 Compared with NEMO (TP3a): the longitudes are **negative** here (−180 to 180), and the time axis uses the usual calendar.

**A regional model lives inside a global one.** This cell is a demonstration: it draws the CROCO domain on the NEMO SST of TP3a. At its **open boundaries** (the edges in the ocean, west and south), CROCO is **told** what comes in (temperature, salinity, currents, sea level) by a global model or a reanalysis: that is how El Niño (TP3a, A.3) gets into it.

In [ ]:
ds_nemo2 = xr.open_dataset(NEMO_2D)
sst_nemo_dec = ds_nemo2["tos"].sel(time_counter="2023-12").squeeze()

fig, ax = plt.subplots(figsize=(13, 6), subplot_kw={"projection": ccrs.PlateCarree(central_longitude=-150)})
sst_nemo_dec.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal, vmin=18, vmax=31,
                  cbar_kwargs={"label": "NEMO SST, December 2023 (°C)", "shrink": 0.7})
# Outline of the CROCO domain: the 4 edges of the grid
edges = [ds.lon_rho.isel(eta_rho=0), ds.lon_rho.isel(xi_rho=-1), ds.lon_rho.isel(eta_rho=-1)[::-1], ds.lon_rho.isel(xi_rho=0)[::-1]]
edges_lat = [ds.lat_rho.isel(eta_rho=0), ds.lat_rho.isel(xi_rho=-1), ds.lat_rho.isel(eta_rho=-1)[::-1], ds.lat_rho.isel(xi_rho=0)[::-1]]
ax.plot(np.concatenate(edges), np.concatenate(edges_lat), color="magenta", linewidth=2, transform=ccrs.PlateCarree())
ax.set_extent([120, 300, -30, 35], crs=ccrs.PlateCarree())
ax.coastlines(linewidth=0.5)
ax.set_title("The CROCO domain (magenta) inside the global NEMO ocean")
plt.show()

---
<a id="part-a"></a>
# A. The surface: the last level

**🎯 Your turn**: select the **surface level** of `temp` and average it over the 3 days, then map it. Remember: in CROCO the surface is the **last** level.

In [ ]:
# TODO: surface level (hint: .isel(s_rho=...)), then mean over time
sst_model = ...

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
sst_model.plot(ax=ax, x="lon_rho", y="lat_rho", transform=ccrs.PlateCarree(),
               cmap=cmocean.cm.thermal, cbar_kwargs={"label": "SST (°C)", "shrink": 0.7})
base_map(ax, "CROCO — SST, mean 1-3 December 2023")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- The surface is the last level: `.isel(s_rho=-1)`.
- Then `.mean(dim="time")`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
sst_model = ds["temp"].isel(s_rho=-1).mean(dim="time")
```

</details>

**✅ Check**: SST between **~26 °C and ~30.5 °C**. Note the **cold water** off the gulfs of **Tehuantepec** (Mexico, ~95°W) and **Papagayo** (Costa Rica/Nicaragua, ~87°W): in December, strong wind jets blow through gaps in the mountains, mix the ocean and bring up cold water (TP1b). Compare with the zoom of TP3a (A.2): the Papagayo cold water was a handful of 1° squares; here it has its real shape. Note also the `x="lon_rho", y="lat_rho"` in `.plot()`: with 2D coordinates, xarray must be told which ones to use.

---
<a id="part-b"></a>
# Part B — Sigma levels vs z-levels

<a id="b-1"></a>
## B.1 Where are the model levels? CROCO vs NEMO

`depth_levels(ds, zeta)` computes the depth (m, negative downwards) of every CROCO level from the bathymetry `h`, the sea level `zeta` and the vertical grid parameters stored in the file (`hc`, `s_rho`, `Cs_rho`):

$$z = \zeta + (\zeta + h)\,\frac{h_c\, s + h\, C_s}{h_c + h}$$

You do not need to remember it: just keep in mind that **the depth of a CROCO level depends on the place** (through `h`), while a NEMO level has the same depth everywhere (TP3a, B.1).

This cell is a **demonstration**: run it and look at the figures. It draws the two vertical grids along the same latitude, **9.5°N** (a row of the NEMO grid), from 92°W to the coast of Costa Rica.

In [ ]:
# CROCO: depth of the 50 sigma levels, first day
z0 = depth_levels(ds, ds["zeta"].isel(time=0))
LAT_GRID = 9.5
z0_sec = section_lat(z0, LAT_GRID)                 # (s_rho, xi_rho): the 50 levels along 9.5°N
h_sec = section_lat(ds["h"], LAT_GRID)             # CROCO sea floor along the same section

# NEMO: the 75 z-levels, and the sea floor of each 1° column (bottom of its deepest wet level)
ds_nemo3 = xr.open_dataset(NEMO_3D)
nemo_col = ds_nemo3["thetao"].isel(time_counter=0).sel(lat=LAT_GRID, lon=slice(-92 + 360, -83 + 360))
interfaces = ds_nemo3["deptht_bnds"].isel(bnds=1)                             # bottom of each layer
nemo_floor = interfaces.where(nemo_col.notnull()).max("deptht")               # sea floor of each column
lon_nemo = nemo_col["lon"].values - 360
x_stairs = np.repeat(np.append(lon_nemo - 0.5, lon_nemo[-1] + 0.5), 2)[1:-1]   # staircase outline
z_stairs = -np.repeat(nemo_floor.values, 2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
axes[0].plot(z0_sec.lon_rho, z0_sec.T, color="0.3", linewidth=0.4)            # one line per level
axes[0].fill_between(h_sec.lon_rho, -h_sec, -6000, color="#d4c9a8")
axes[0].set_facecolor("#d4c9a8")                                          # land behind the coast
axes[0].set_title("CROCO (~5.5 km): 50 sigma levels, they follow the sea floor")
axes[1].hlines(-interfaces, -92.5, -83, color="0.3", linewidth=0.4)            # one line per level
axes[1].fill_between(x_stairs, z_stairs, -6000, color="#d4c9a8")
axes[1].set_title("NEMO (1°): 75 z-levels, the sea floor is a staircase")
for ax in axes:
    ax.set_xlim(-92.5, -83.5)
    ax.set_ylim(-4500, 0)
    ax.set_xlabel("Longitude (°E)")
axes[0].set_ylabel("Depth (m)")
fig.suptitle(f"The two vertical grids along {LAT_GRID}°N")
plt.show()

# Zoom on the upper 300 m, near the coast
for ax in axes:
    ax.set_xlim(-87, -83.5)
    ax.set_ylim(-300, 0)
display(fig)

# Number of levels in the water column at three places
print("                               CROCO: depth, levels above 50 m | NEMO: depth of the 1° cell, levels above 50 m")
for name, lon, lat in [("Costa Rica Dome    ", -89.0, 9.0),
                       ("Gulf of Papagayo   ", -85.9, 10.6),
                       ("Gulf of Panama     ", -79.5, 8.5)]:
    z_col = nearest_point(z0, lon, lat)
    h_col = float(nearest_point(ds["h"], lon, lat))
    n_col = ds_nemo3["thetao"].isel(time_counter=0).sel(lon=lon + 360, lat=lat, method="nearest")
    n_floor = float(interfaces.where(n_col.notnull()).max())
    print(f"{name} ({lon}, {lat}): {h_col:6.0f} m, {int((z_col > -50).sum()):2d} levels"
          f"   |   {n_floor:6.0f} m, {int((n_col.notnull() & (n_col.deptht < 50)).sum()):2d} levels")

**🎯 Look and answer** (no code): from the figures and the table,
1. Offshore (west of 88°W), where are the CROCO levels dense? And the NEMO levels?
2. Near the coast (zoom), what happens to the CROCO levels? To the NEMO levels?
3. In the Gulf of Papagayo, the real sea floor is ~90 m deep. How many levels does each model have above 50 m there? And how deep is the sea floor of the NEMO cell?

**✅ Check**:
1. Offshore, both grids are **dense near the surface** (CROCO: ~10 levels in the upper 50 m; NEMO: 18), and sparse at depth. On the open ocean, the two grids are not so different.
2. Near the coast, the 50 CROCO levels **squeeze** into the shallower water column: on the slope and the shelf, they are only a few metres apart. NEMO's levels stay horizontal: the sea floor cuts them, as a **staircase** of 1° steps. The deep trench at 86°W (the Middle America Trench) and the steep slope are lost, and the last NEMO column (84°W, 1° wide, mostly over land in reality) is still ~800 m deep.
3. Gulf of Papagayo: CROCO sees a **91 m** deep shelf with all its **50 levels**, **35 of them above 50 m**; the NEMO cell there is **~3200 m** deep: in a 1° cell the shelf does not exist. Gulf of Panama: CROCO 52 m, with 49 of its 50 levels above 50 m; NEMO a ~100 m deep cell with 18 levels above 50 m — always 18, wherever the water is deep enough: z-levels do not adapt to the place.

👉 **Sigma levels** keep a fine vertical resolution everywhere, even on a 50 m shelf: that's why they are used for **coastal and regional** models (upwelling, tides, shelf processes). **z-levels** are simpler (a level is a fixed depth) and well suited to the **open ocean** and to global models. Their weak point is the coast and the steep slopes.

---
<a id="b-2"></a>
## B.2 A vertical section across the Costa Rica Dome

**🎯 Your turn**: build the vertical section of temperature along 9°N, 3-day mean. Each point of a model section has its own depth: we attach the depths to the section as a coordinate `z`, and xarray draws it with `.plot.contourf(x="lon_rho", y="z")`.

In [ ]:
# TODO: 3-day means of temp (3D), then the depths of the levels for the 3-day mean sea level
temp_mean = ...
z_mean = ...

# TODO: section of temp_mean along 9°N (hint: section_lat)
temp_sec = ...
temp_sec = temp_sec.assign_coords(z=section_lat(z_mean, 9.0))   # the depth of every point of the section
h_sec = section_lat(ds["h"], 9.0)

fig, ax = plt.subplots(figsize=(12, 5))
temp_sec.plot.contourf(ax=ax, x="lon_rho", y="z", levels=np.arange(8, 31, 1), cmap=cmocean.cm.thermal,
                       extend="both", cbar_kwargs={"label": "Temperature (°C)"})
temp_sec.plot.contour(ax=ax, x="lon_rho", y="z", levels=[20], colors="k", linewidths=2)
ax.fill_between(h_sec.lon_rho, -h_sec, -6000, color="#d4c9a8")   # sea floor
ax.set_ylim(-300, 0)
ax.set_xlim(-104, -83)
ax.set_facecolor("#d4c9a8")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Depth (m)")
ax.set_title("CROCO — temperature along 9°N, mean 1-3 December 2023 (black: 20 °C)")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `ds["temp"].mean(dim="time")`; depths: `depth_levels(ds, ds["zeta"].mean(dim="time"))`.
- `section_lat(temp_mean, 9.0)`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
temp_mean = ds["temp"].mean(dim="time")
z_mean = depth_levels(ds, ds["zeta"].mean(dim="time"))
temp_sec = section_lat(temp_mean, 9.0)
```

</details>

**✅ Check**: a warm surface layer (> 27 °C, ~30-40 m thick) above a sharp thermocline. The **20 °C isotherm** stays between **~35 and ~65 m**, with two domes where it rises to **~35 m**: near 100°W and near **88-89°W** — the **Costa Rica Dome** — then it deepens again to ~65 m towards the coast. Same structure as the WOA climatology in TP1b, but with much finer details (5 km vs 1°) and for 3 given days (not a long-term mean).

💬 Compare with TP3a (B.2): the NEMO section had 9 columns of 1° and horizontal level lines; here, ~200 columns, and the levels bend with the sea floor near the coast.

---
<a id="b-3"></a>
## B.3 A map at 100 m: interpolate, not select

In NEMO (TP3a, B.3), a map at 100 m was one line: `.sel(deptht=100, method="nearest")`, because each level has a fixed depth. In CROCO, there is **no level at 100 m**: in each column, 100 m falls between two different levels. We must **interpolate vertically**, column by column: that's what `at_depth(da, z, depth)` does (depth in m, **negative**). Where the sea floor is shallower than the requested depth, the result is `NaN`.

**🎯 Your turn**: map the temperature at **100 m** (3-day mean), from `temp_mean` and `z_mean` (B.2).

In [ ]:
# TODO: temperature at 100 m, from temp_mean and z_mean (B.2)
t100 = ...

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
t100.plot(ax=ax, x="lon_rho", y="lat_rho", transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal,
          cbar_kwargs={"label": "Temperature at 100 m (°C)", "shrink": 0.7})
ax.contour(ds["lon_rho"], ds["lat_rho"], ds["h"], levels=[100], colors="k", linewidths=0.5, transform=ccrs.PlateCarree())
base_map(ax, "CROCO — temperature at 100 m, mean 1-3 December 2023")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `at_depth(temp_mean, z_mean, -100)` — note the minus sign.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
t100 = at_depth(temp_mean, z_mean, -100)
```

</details>

**✅ Check**: much larger contrasts than at the surface (~13 to ~27 °C, vs 26 to 30.5 °C): **~13-15 °C along a band at 7-10°N** (the thermocline ridge, which includes the Costa Rica Dome), and **~26 °C in the core of the big eddy at 13°N, 100°W** (an anticyclone pushes the thermocline down). Areas shallower than 100 m (inside the thin black line, the 100 m isobath) are blank: there is no water at 100 m there.

💬 `.sel(s_rho=...)` would **not** give a map at 100 m: it would give one sigma level, at 30 m on the shelf and 1500 m offshore. **With sigma levels, a fixed depth always means interpolation.**

---
<a id="advanced-a"></a>
### 🚀 Advanced track A — the depth of the thermocline

Map the **depth of the 20 °C isotherm** (3-day mean), a common proxy for the thermocline depth. It's `at_depth` the other way round: in each column, interpolate the **depth** at the **temperature** 20 °C (hint: copy `at_depth` and swap the roles of `z` and the temperature). Where is the thermocline shallowest?

In [ ]:
# 🚀 Your code here

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Write a function for **one** column, `def depth_of_20C(t_col, z_col): return np.interp(20, t_col, z_col, left=np.nan, right=np.nan)`, then repeat it everywhere with `xr.apply_ufunc(depth_of_20C, temp_mean.compute(), z_mean.compute(), input_core_dims=[["s_rho"], ["s_rho"]], vectorize=True)`.
- In a column the temperature increases from the bottom (level 0) to the surface, as `np.interp` needs.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
def depth_of_20C(t_col, z_col):
    """In one water column: depth (m) where the temperature is 20 °C (temperature increases upwards)."""
    return np.interp(20, t_col, z_col, left=np.nan, right=np.nan)

z20 = xr.apply_ufunc(depth_of_20C,
                     temp_mean.compute(), z_mean.compute(),     # load the data in memory
                     input_core_dims=[["s_rho"], ["s_rho"]],    # one column = along s_rho
                     vectorize=True)                            # repeat for every grid point

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
(-z20).plot(ax=ax, x="lon_rho", y="lat_rho", transform=ccrs.PlateCarree(), cmap="viridis_r", vmin=0, vmax=120,
            cbar_kwargs={"label": "Depth of the 20 °C isotherm (m)", "shrink": 0.7})
base_map(ax, "CROCO — depth of the 20 °C isotherm, mean 1-3 December 2023")
plt.show()
print(f"Costa Rica Dome (89°W, 9°N): {-float(nearest_point(z20, -89, 9)):.0f} m   "
      f"eddy (100°W, 13°N): {-float(nearest_point(z20, -100, 13)):.0f} m")

# Shallowest (~35-40 m) in the Costa Rica Dome (~9°N, 89°W), along the ridge at 7-9°N
# west of 95°W, and off Tehuantepec (~14°N, 95°W). Deepest (~130 m) in the core of the
# anticyclonic eddy at 13°N, 100°W.
```

</details>

**✅ Check (advanced)**: shallowest (~35-40 m) in the **Costa Rica Dome** (~9°N, 89°W), along the ridge at 7-9°N west of 95°W, and off Tehuantepec (~14°N, 95°W); deepest (~130 m) in the core of the anticyclonic eddy at 13°N, 100°W. Compare with the NEMO map of TP3a (advanced track B): same ridge and Dome, with much finer details.

---
<a id="advanced-b"></a>
### 🚀 Advanced track B — regional model vs global reanalysis: the Dome section

**🎯 Your turn**: build the same temperature section along 9°N from **GLORYS12** (3-day mean, `thetao`) and compare it with CROCO (B.2), over the same longitudes. GLORYS12 uses **z-levels**, like NEMO: its section is a simple selection.

In [ ]:
ds_glo = xr.open_dataset(GLORYS_FILE)
DAYS = slice("2023-12-01", "2023-12-03")      # the 3 days of the CROCO file
# TODO: GLORYS temperature: same 3 days, mean over time, latitude nearest to 9°N
glo_sec = ...

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
levels = np.arange(8, 31, 1)
# CROCO: section of B.2 (z negative)
cf = temp_sec.plot.contourf(ax=axes[0], x="lon_rho", y="z", levels=levels, cmap=cmocean.cm.thermal,
                            extend="both", add_colorbar=False)
temp_sec.plot.contour(ax=axes[0], x="lon_rho", y="z", levels=[20], colors="k", linewidths=2)
# GLORYS: depth is positive downwards -> use -depth to match CROCO
axes[1].contourf(glo_sec.longitude, -glo_sec.depth, glo_sec, levels=levels, cmap=cmocean.cm.thermal, extend="both")
axes[1].contour(glo_sec.longitude, -glo_sec.depth, glo_sec, levels=[20], colors="k", linewidths=2)
axes[0].set_ylim(-300, 0)
axes[0].set_ylabel("Depth (m)")
for ax in axes:
    ax.set_xlim(-92, -83.5)                # Pacific side, up to the coast
    ax.set_facecolor("#d4c9a8")
    ax.set_xlabel("Longitude (°E)")
axes[0].set_title("CROCO (regional, free run, ~5 km)")
axes[1].set_title("GLORYS12 (global reanalysis, 1/12°)")
fig.colorbar(cf, ax=axes, label="Temperature (°C), 9°N, 1-3 Dec 2023")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `ds_glo["thetao"].sel(time=DAYS).mean(dim="time")`, then `.sel(latitude=9.0, method="nearest")`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
glo_sec = ds_glo["thetao"].sel(time=DAYS).mean(dim="time").sel(latitude=9.0, method="nearest")
```

</details>

**✅ Check (advanced)**: both show the thermocline rising around **88-89°W** (Costa Rica Dome) and deepening towards the coast, but the Dome is **sharper in CROCO** (20 °C isotherm from ~60 m at 92°W up to ~35 m at 88.5°W, then ~65 m at the coast) than in GLORYS (flatter, ~42-57 m).

⚠️ A convention to watch in a model/reanalysis comparison: CROCO depths are **negative** (z, counted upwards), GLORYS depths are **positive** (`depth`, counted downwards) — hence the `-glo_sec.depth` to put both on the same axis.

---
<a id="c-debugging"></a>
## C. 🐞 Debugging exercise

A colleague wants the map of the model's sea surface temperature on 1 December. *"It crashes — and once I got it to run, the map looked strange, with temperatures of 2 °C in the middle of the tropics!"* The cell contains **three bugs**: two raise an error, one is **silent**.

Run it, read the error message (the last line), fix, run again… until the map looks like A.1.

In [ ]:
# 🐞 Fix me! (3 bugs)
sst_bug = ds["temperature"].isel(time=0, s_rho=0)

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
sst_bug.plot(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal)
base_map(ax, "CROCO — SST, 1 December 2023")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Read the last line of each error: check `list(ds.data_vars)` and `ds.coords`.
- A map at 2 °C in the tropics is not the surface… which level is `s_rho=0`? (section 1)

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
# Fixed version
sst_bug = ds["temp"].isel(time=0, s_rho=-1)

fig, ax = plt.subplots(figsize=(11, 7), subplot_kw={"projection": ccrs.PlateCarree()})
sst_bug.plot(ax=ax, x="lon_rho", y="lat_rho", transform=ccrs.PlateCarree(), cmap=cmocean.cm.thermal)

# Bug 1: KeyError 'temperature'      -> the variable is called temp (list(ds.data_vars)).
# Bug 2: 'lon' / 'lat' do not exist -> the 2D coordinates are lon_rho / lat_rho.
# Bug 3 (silent): s_rho=0 is the BOTTOM level (1-4 °C in the deep ocean, warm on the
#        shelves where the bottom is shallow); the surface is s_rho=-1.
```

</details>

**✅ Check**: the fixed cell gives the same map as in Part A (26-30.5 °C). If your map shows **1-4 °C offshore and warm water only along the coasts**, you are still plotting the **bottom** level.

---
<a id="d-recap"></a>
## D. Recap, in pairs: global or regional?

1. Go back to the recap table at the end of **TP3a** (section D) and fill in the **CROCO column**.
2. For each question below, would you use a **global** model (like NEMO), a **regional** model (like CROCO), or both? Why? Write one sentence each (you will reuse them in the mini-project):
   - a. How did the 2023-24 El Niño change the sea surface temperature off Costa Rica?
   - b. Where and when does cold water reach the coast in the Gulf of Papagayo?
   - c. Low oxygen on the shelf of the Gulf of Nicoya.
   - d. The temperature of the eastern Pacific in 2100, under a climate scenario.

---
## To go further

- CROCO documentation (grid, vertical coordinates, outputs): <https://croco-ocean.gitlabpages.inria.fr/croco_doc/>
- NEMO documentation: <https://www.nemo-ocean.eu>
- Next: **TP4 mini-project** — choose a region and compare models, reanalysis and observations.